# Notebook 1 — JAX Crash Course

In this notebook, we introduce the minimal JAX concepts that will be used throughout the hands-on sessions.

The goal is not to learn all of JAX, but to understand how JAX differs from ordinary Python and NumPy.

We will focus on:

- `jax.numpy`
- immutable arrays
- pure functions
- automatic differentiation with `jax.grad`
- just-in-time compilation with `jax.jit`
- vectorisation with `jax.vmap`
- functional random numbers
- simple functional loops with `jax.lax.scan`

The next notebook will use these ingredients to build a minimal Variational Monte Carlo implementation from scratch.

## The design philosophy of JAX

JAX was designed around a simple idea:

> Numerical programs should be written as pure mathematical functions that can be automatically transformed.

This differs from traditional scientific Python, where programs often rely on mutable state, side effects, and imperative execution.

Several important design choices follow from this philosophy:

### Pure functions

Functions should depend only on their inputs and return their outputs explicitly.

```python
y = f(x)
```

rather than modifying global variables or hidden state.

### Immutable data

Arrays are never modified in place. Instead, operations return new arrays. This makes program transformations easier and more predictable. Transformations as first-class objects. JAX can automatically transform functions through:

- `jax.grad`: automatic differentiation
- `jax.jit`: compilation
- `jax.vmap`: automatic vectorisation
- `jax.pmap`: parallel execution

These transformations can often be combined with a single line of code.

### Explicit randomness

Random number generation is handled through explicit PRNG keys rather than a hidden global random state. This improves reproducibility and parallelisation.

### Accelerator-first design

The same code can run on CPUs, GPUs, and TPUs with little or no modification. As a consequence, writing efficient JAX code often requires a slightly different mindset than writing ordinary NumPy code. Throughout these tutorials, we will gradually adopt this functional style of programming.

In [1]:
import jax
import jax.numpy as jnp
import numpy as np

## 1. JAX looks like NumPy

JAX provides a NumPy-like interface through `jax.numpy`.

For many simple array operations, replacing

```python
import numpy as np
```

by 

```python
import jax.numpy as jnp
```

is enough.

However, JAX arrays are not NumPy arrays. They obey a different execution model, designed for automatic differentiation, compilation, vectorisation, and accelerator hardware.

In [2]:
x_np = np.arange(5)
x_jax = jnp.arange(5)

print(x_np)
print(x_jax)

[0 1 2 3 4]
[0 1 2 3 4]


In [3]:
print(type(x_np))
print(type(x_jax))

<class 'numpy.ndarray'>
<class 'jaxlib._jax.ArrayImpl'>


The syntax of `jax.numpy` is intentionally very similar to NumPy.

Most common operations work exactly as expected.

In [4]:
x = jnp.arange(10)

print("Array:")
print(x)

print("\nMean:")
print(jnp.mean(x))

print("\nVariance:")
print(jnp.var(x))

print("\nEuclidean norm:")
print(jnp.linalg.norm(x))

Array:
[0 1 2 3 4 5 6 7 8 9]

Mean:
4.5

Variance:
8.25

Euclidean norm:
16.881943


## 2. Arrays are immutable

One of the most important differences between NumPy and JAX is that JAX arrays are immutable.

In NumPy, modifying an array in-place is perfectly normal:

```python
x[0] = 42
```

In JAX, this operation is forbidden. This design choice makes it possible for JAX to safely apply transformations such as automatic differentiation and compilation. This design also encourages an array-oriented programming style that maps efficiently to massively parallel hardware such as GPUs.

In [5]:
x = jnp.arange(5)

try:
    x[0] = 42
except Exception as e:
    print(type(e).__name__)
    print(e)

TypeError
JAX arrays are immutable and do not support in-place item assignment. Instead of x[idx] = y, use x = x.at[idx].set(y) or another .at[] method: https://docs.jax.dev/en/latest/_autosummary/jax.numpy.ndarray.at.html


Instead, JAX provides functional update operations.

Rather than modifying an array, a new array is returned.

In [6]:
x = jnp.arange(5)

y = x.at[0].set(42)

print("Original array:")
print(x)

print("\nUpdated array:")
print(y)

Original array:
[0 1 2 3 4]

Updated array:
[42  1  2  3  4]


## 3. Automatic differentiation

One of the main motivations for using JAX is its ability to automatically compute derivatives. Given a Python function, JAX can construct another function that evaluates its gradient. This is particularly useful in machine learning and variational optimisation, where gradients are needed to update model parameters.

In [9]:
def f(x):
    return x**2

jax.grad(f)(3.0)

Array(6., dtype=float32, weak_type=True)

The derivative of $f(x)=x^2$ is $f'(x)=2x$. Evaluating the derivative at $x=3$ therefore gives $6$.

In [12]:
def f(x):
    return jnp.sin(x) + x**2

x = 1.0

print("f(x) =", f(x))
print("f'(x) =", jax.grad(f)(x))
print("cos(1) + 2*1 = ", jnp.cos(1) + 2)

f(x) = 1.841471
f'(x) = 2.5403023
cos(1) + 2*1 =  2.5403023


In [13]:
def f(x):
    return jnp.sum(x**2)

x = jnp.array([1.0, 2.0, 3.0])

jax.grad(f)(x)

Array([2., 4., 6.], dtype=float32)

Gradients can also be computed with respect to arbitrary function arguments. By default, `jax.grad` differentiates with respect to the first argument, but this can be changed using the `argnums` keyword.

In [15]:
def f(x, y):
    return x**2 + x * y + y**2

print("df/dx =", jax.grad(f, argnums=0)(1.0, 2.0))
print("df/dy =", jax.grad(f, argnums=1)(1.0, 2.0))

df/dx = 4.0
df/dy = 5.0


In practice, we will often think of a function as depending on some parameters and some configurations,

```python
log_psi(params, samples)
```

and differentiate with respect to the parameters.

In [16]:
def loss(w, x):
    return jnp.sum((w * x) ** 2)

x = jnp.array([1.0, 2.0, 3.0])

jax.grad(loss)(2.0, x)

Array(56., dtype=float32, weak_type=True)

## 4. Just-in-time compilation

Besides automatic differentiation, another key feature of JAX is the ability to compile numerical code. Compilation can significantly accelerate repeated function evaluations, especially for large computations running on GPUs.

In [18]:
def f(x):
    return jnp.sum(jnp.sin(x) ** 2)

x = jnp.arange(1_000_000)

f(x)

Array(500000.06, dtype=float32)

In [19]:
f_jit = jax.jit(f)

f_jit(x)

Array(500000.1, dtype=float32)

Unlike traditional Python optimisation libraries, `jax.jit` does not execute a function. Instead, it transforms a function into a new compiled function. This idea of function transformations is central to JAX.

For example:

- `jax.grad` transforms a function into its derivative
- `jax.jit` transforms a function into a compiled version
- `jax.vmap` transforms a function into a vectorised version

The transformed function can then be used exactly like the original one.

In [20]:
def f(x):
    return jnp.sin(x) + x**2

df = jax.grad(f)
f_jit = jax.jit(f)

print(f(1.0))
print(df(1.0))
print(f_jit(1.0))

1.841471
2.5403023
1.841471


Compilation does not happen when `jax.jit` is called. Instead, JAX traces the function during its first execution, builds an optimised computation graph, compiles it, and caches the result. As a consequence, the first call is often slower than subsequent calls.

In [21]:
f_jit = jax.jit(f)

In [27]:
%time f_jit(x).block_until_ready()

CPU times: user 5.13 ms, sys: 1.43 ms, total: 6.56 ms
Wall time: 1.52 ms
CPU times: user 4.64 ms, sys: 300 μs, total: 4.94 ms
Wall time: 1.29 ms


Array([ 0.0000000e+00,  1.8414710e+00,  4.9092975e+00, ...,
       -7.3337997e+08, -7.3137997e+08, -7.2937997e+08], dtype=float32)

In [23]:
%time f_jit(x).block_until_ready()

CPU times: user 5.51 ms, sys: 1.04 ms, total: 6.55 ms
Wall time: 1.5 ms


Array([ 0.0000000e+00,  1.8414710e+00,  4.9092975e+00, ...,
       -7.3337997e+08, -7.3137997e+08, -7.2937997e+08], dtype=float32)

The first call includes tracing and compilation overhead. Subsequent calls reuse the compiled version and are typically much faster. For long-running computations, the compilation cost is usually negligible compared to the execution time savings.

## 5. Vectorisation with `jax.vmap`

Scientific computations often involve applying the same function to many inputs. A common approach in Python is to use a loop.

In [28]:
def square(x):
    return x**2

xs = jnp.arange(5)

jnp.array([square(x) for x in xs])

Array([ 0,  1,  4,  9, 16], dtype=int32)

While this works, Python loops become a bottleneck for large computations. JAX instead encourages an array-oriented style where operations are applied to whole batches simultaneously. The transformation `jax.vmap` automatically converts a function acting on a single input into a function acting on a batch of inputs.

In [29]:
square_vmap = jax.vmap(square)

square_vmap(xs)

Array([ 0,  1,  4,  9, 16], dtype=int32)

Importantly, we did not modify the original function. The function ```square``` still acts on a single number, while ```jax.vmap(square)``` creates a new function acting on a batch. 

Vectorisation becomes particularly useful when evaluating functions on large collections of inputs. This pattern appears frequently in machine learning, Monte Carlo methods, and scientific computing.

In [30]:
def f(x):
    return jnp.sin(x) + x**2

f_batch = jax.vmap(f)

xs = jnp.linspace(-2.0, 2.0, 10)

f_batch(xs)

Array([ 3.0907025 ,  1.4198692 ,  0.33837575, -0.1739254 , -0.17101505,
        0.26978055,  1.0628145 ,  2.1307602 ,  3.419637  ,  4.9092975 ],      dtype=float32)

Function transformations can naturally be composed:

In [31]:
f_batch_jit = jax.jit(jax.vmap(f))

f_batch_jit(xs)

Array([ 3.0907025 ,  1.4198693 ,  0.33837578, -0.17392538, -0.17101505,
        0.26978055,  1.0628145 ,  2.1307602 ,  3.419637  ,  4.9092975 ],      dtype=float32)

The ability to compose transformations such as

```python
jax.grad(...)
jax.jit(...)
jax.vmap(...)
```

is one of the central ideas behind JAX.

## 6. Random numbers

Random number generation is one of the areas where JAX differs most significantly from NumPy.

In NumPy, random numbers are typically generated from a hidden global state:

```python
np.random.randn()
```

In JAX, randomness is explicit. Random numbers are generated from a pseudo-random number generator (PRNG) key that is passed as an argument to functions. This design improves reproducibility and enables efficient parallel execution.

In [32]:
key = jax.random.PRNGKey(42)

key

Array([ 0, 42], dtype=uint32)

Random numbers are generated from a key:

In [33]:
jax.random.normal(key)

Array(-0.02830462, dtype=float32)

Unlike NumPy, keys should not be reused. Instead, a key is split into new independent keys.

In [35]:
key, subkey = jax.random.split(key)

print(key)
print(subkey)

jax.random.normal(subkey)

[1012194634 3152801799]
[1705926158  899080142]


Array(-0.21089035, dtype=float32)

A common pattern is therefore to create a top-level `key` acting as a seed, and splitting subkeys

```python
key, subkey = jax.random.split(key)
```

for the needed random operations.

In [36]:
key = jax.random.PRNGKey(0)

for _ in range(3):
    key, subkey = jax.random.split(key)
    print(jax.random.normal(subkey))

-2.4424558
-1.2574776
-1.3877681


Random arrays can be generated in exactly the same way.

In [37]:
key, subkey = jax.random.split(key)

jax.random.normal(subkey, shape=(5,))

Array([-2.3022664 ,  0.05277798,  1.6845714 , -0.4193235 , -0.07544231],      dtype=float32)

Because randomness is explicit, functions that depend on random numbers typically receive a key as an input and return an updated key as part of their output. This functional style will appear repeatedly in Monte Carlo algorithms.

In [40]:
def sample_uniform(key):
    key, subkey = jax.random.split(key)
    x = jax.random.uniform(subkey)
    return key, x

key = jax.random.PRNGKey(0)

key, x = sample_uniform(key)
print(x)

key, x = sample_uniform(key)
print(x)

0.0072938204
0.104290366


The key idea is that random number generation is no longer hidden. Randomness becomes an explicit part of the computation.

## Summary

In this notebook, we introduced some core ideas behind JAX. The main take-home message is that, in JAX, pure functions can be transformed, and these transformations can be freely combined to build efficient and differentiable numerical programs.

In the next notebook, we will use these tools to implement a minimal Variational Monte Carlo algorithm from scratch.